In [0]:
%run "../../commons/commons_imports"

In [0]:
df_item_bronze= read(
    base_path=BRONZE_PATH,
    table_name=TS_ITEM,
    recursive_by_year=False,
    format = "delta",
)

In [0]:
df_item_silver = (

    df_item_bronze

    # =====================================================
    # Região
    # =====================================================

    .withColumn(
        "REGIAO",
        when(col("SG_UF").isin("AC","AP","AM","PA","RO","RR","TO"), "Norte")
        .when(col("SG_UF").isin("AL","BA","CE","MA","PB","PE","PI","RN","SE"), "Nordeste")
        .when(col("SG_UF").isin("DF","GO","MT","MS"), "Centro-Oeste")
        .when(col("SG_UF").isin("ES","MG","RJ","SP"), "Sudeste")
        .when(col("SG_UF").isin("PR","RS","SC"), "Sul")
    )

    # =====================================================
    # Faixa de dificuldade (Parâmetro B)
    # =====================================================

    .withColumn(
        "FAIXA_DIFICULDADE_ITEM",
        when(col("NU_PARAM_B").isNull(), "Não informado")
        .when(col("NU_PARAM_B") < 650, "Muito Fácil")
        .when(col("NU_PARAM_B") < 700, "Fácil")
        .when(col("NU_PARAM_B") < 750, "Médio")
        .when(col("NU_PARAM_B") < 800, "Difícil")
        .otherwise("Muito Difícil")
    )

    # =====================================================
    # Faixa de discriminação (Parâmetro A)
    # =====================================================

    .withColumn(
        "FAIXA_DISCRIMINACAO",
        when(col("NU_PARAM_A").isNull(), "Não informado")
        .when(col("NU_PARAM_A") < 0.35, "Muito Baixa")
        .when(col("NU_PARAM_A") < 0.65, "Baixa")
        .when(col("NU_PARAM_A") < 1.00, "Boa")
        .otherwise("Muito Boa")
    )

    # =====================================================
    # Faixa do acerto ao acaso (Parâmetro C)
    # =====================================================

    .withColumn(
        "FAIXA_ACERTO_AO_ACASO",
        when(col("NU_PARAM_C").isNull(), "Não informado")
        .when(col("NU_PARAM_C") < 0.15, "Baixa")
        .when(col("NU_PARAM_C") < 0.25, "Média")
        .otherwise("Alta")
    )

    # =====================================================
    # Possui parâmetros politômicos
    # =====================================================

    .withColumn(
        "IN_PARAMETROS_POLITOMICOS",
        when(
            col("NU_PARAM_B1").isNotNull(),
            1
        ).otherwise(0)
    )

    # =====================================================
    # Quantidade de parâmetros B
    # =====================================================

    .withColumn(
        "QT_PARAMETROS_B",
        (
            when(col("NU_PARAM_B1").isNotNull(), 1).otherwise(0)
            + when(col("NU_PARAM_B2").isNotNull(), 1).otherwise(0)
            + when(col("NU_PARAM_B3").isNotNull(), 1).otherwise(0)
            + when(col("NU_PARAM_B4").isNotNull(), 1).otherwise(0)
        )
    )

    # =====================================================
    # Auditoria
    # =====================================================

    .withColumn(
        "ANO_CARGA",
        year(col("TS_PROCESSAMENTO"))
    )

    .withColumn(
        "MES_CARGA",
        month(col("TS_PROCESSAMENTO"))
    )

)

In [0]:
df_item_silver_selected = df_item_silver.select(

    # ============================================
    # Chave Técnica
    # ============================================
    "SK_ITEM",

    # ============================================
    # Chaves de Negócio
    # ============================================
    "CO_ITEM",
    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",

    "CO_UF",
    "SG_UF",
    "REGIAO",

    "CO_BLOCO",
    "NU_POSICAO",

    # ============================================
    # Informações do Item
    # ============================================
    "TP_SERIE",
    "DS_SERIE",

    "TP_DISCIPLINA",
    "DS_DISCIPLINA",

    "NU_DESCRITOR_HABILIDADE",

    "DS_GABARITO",

    "TP_RESPOSTA_ITEM",
    "DS_RESPOSTA_ITEM",

    "TP_MODELO_TRI",
    "DS_MODELO_TRI",

    "IN_ITEM_COMUM",
    "DS_ITEM_COMUM",

    # ============================================
    # Parâmetros TRI
    # ============================================
    "NU_PARAM_A",
    "FAIXA_DISCRIMINACAO",

    "NU_PARAM_B",
    "FAIXA_DIFICULDADE_ITEM",

    "NU_PARAM_C",
    "FAIXA_ACERTO_AO_ACASO",

    "NU_PARAM_B1",
    "NU_PARAM_B2",
    "NU_PARAM_B3",
    "NU_PARAM_B4",

    "IN_PARAMETROS_POLITOMICOS",
    "QT_PARAMETROS_B",

    # ============================================
    # Auditoria
    # ============================================
    "ANO_CARGA",
    "MES_CARGA",

    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
validate_primary_key(df_item_silver_selected, "SK_ITEM")

validate_not_null(
    df_item_silver_selected,
    [
        "SK_ITEM"
    ]
)

validate_years(df_item_silver_selected)

In [0]:
write_delta(
    df=df_item_silver_selected,
    base_path=SILVER_PATH,
    table_name=TS_ITEM,
    merge_keys=["SK_ITEM"]
)